In [18]:
import torch
import torch.nn as nn
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torchvision.ops as ops

In [19]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

epochs = 25
batch_size = 100
learning_rate = 0.001

transform = transforms.Compose([
    transforms.Pad(4),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32),
    transforms.ToTensor()
])

train_dataset = datasets.CIFAR10(root='../../data', train=True, transform=transform, download=True)
test_dataset = datasets.CIFAR10(root='../../data', train=False, transform=transforms.ToTensor())

train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)


In [20]:
def conv3x3(in_channels, out_channels, stride=1):
    return nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=3, padding=1, stride=stride,
                     bias=False)


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, downsample=None, drop_prob=0.0):
        super().__init__()
        self.conv1 = conv3x3(in_channels, out_channels, stride)
        self.batchnorm1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(out_channels, out_channels)
        self.batchnorm2 = nn.BatchNorm2d(out_channels)
        self.downsample = downsample
        self.drop_path = ops.StochasticDepth(p=drop_prob, mode='row')

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.batchnorm1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.batchnorm2(out)

        out = self.drop_path(out)

        if self.downsample:
            residual = self.downsample(x)

        out += residual
        out = self.relu(out)

        return out


In [21]:
class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=10, max_drop_prob=0.1):
        super().__init__()
        self.current_block=0
        self.max_drop_prob = max_drop_prob
        self.total_block=sum(layers)
        self.in_channels = 16
        self.conv = conv3x3(3, 16)
        self.batchnorm = nn.BatchNorm2d(16)
        self.relu = nn.ReLU(inplace=True)
        self.layer1 = self.create_layer(block, 16, layers[0])
        self.layer2 = self.create_layer(block, 32, layers[1], 2)
        self.layer3 = self.create_layer(block, 64, layers[2], 2)
        self.avg_pool = nn.AvgPool2d(kernel_size=8)
        self.fc = nn.Linear(64, num_classes)

    def create_layer(self, block, out_channels, blocks, stride=1):
        downsample = None
        if (stride != 1) or (self.in_channels != out_channels):
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels))
        layers = []
        self.current_block += 1
        layers.append(block(self.in_channels, out_channels, stride, downsample, self.calculate_drop_prob()))
        self.in_channels = out_channels

        for i in range(1, blocks):
            self.current_block += 1
            layers.append(block(out_channels, out_channels, drop_prob=self.calculate_drop_prob()))

        self.in_channels = out_channels

        return nn.Sequential(*layers)

    def calculate_drop_prob(self):
        return (self.current_block / self.total_block) * self.max_drop_prob

    def forward(self, x):
        out = self.conv(x)
        out = self.batchnorm(out)
        out = self.relu(out)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.avg_pool(out)
        out = out.view(out.size(0), -1)
        out = self.fc(out)

        return out


In [22]:
model = ResNet(ResidualBlock, [2, 2, 2]).to(device).float()
loss_func = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=learning_rate)

decay = 0
model.train()
for epoch in range(epochs):
    if (epoch + 1) % 20 == 0:
        decay += 1
        optimizer.param_groups[0]['lr'] = learning_rate * (0.5 ** decay)

    for i, (inputs, labels) in enumerate(train_loader):
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        loss = loss_func(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (i + 1) % 100 == 0:
        print("Epoch [{}/{}], Step [{}/{}] Loss: {:.4f}"
              .format(epoch + 1, epochs, i + 1, len(train_loader), loss.item()))


Epoch [1/25], Step [500/500] Loss: 1.2239
Epoch [2/25], Step [500/500] Loss: 0.9823
Epoch [3/25], Step [500/500] Loss: 1.0515
Epoch [4/25], Step [500/500] Loss: 0.9971
Epoch [5/25], Step [500/500] Loss: 0.8656
Epoch [6/25], Step [500/500] Loss: 0.7275
Epoch [7/25], Step [500/500] Loss: 0.8760
Epoch [8/25], Step [500/500] Loss: 0.7618
Epoch [9/25], Step [500/500] Loss: 0.9163
Epoch [10/25], Step [500/500] Loss: 0.5468
Epoch [11/25], Step [500/500] Loss: 0.6880
Epoch [12/25], Step [500/500] Loss: 0.6216
Epoch [13/25], Step [500/500] Loss: 0.6849
Epoch [14/25], Step [500/500] Loss: 0.5091
Epoch [15/25], Step [500/500] Loss: 0.5522
Epoch [16/25], Step [500/500] Loss: 0.5850
Epoch [17/25], Step [500/500] Loss: 0.5567
Epoch [18/25], Step [500/500] Loss: 0.4875
Epoch [19/25], Step [500/500] Loss: 0.7098
Epoch [20/25], Step [500/500] Loss: 0.2946
Epoch [21/25], Step [500/500] Loss: 0.6078
Epoch [22/25], Step [500/500] Loss: 0.5263
Epoch [23/25], Step [500/500] Loss: 0.4139
Epoch [24/25], Step 

In [23]:
model.eval()
with torch.no_grad():
    correct = 0
    total = 0

    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        (_, predicted) = torch.max(outputs.data, 1)
        correct += (predicted == labels).sum()

        loss = loss_func(outputs, labels)
        total += labels.size(0)

    print('Accuracy of the model on the test images: {} %'.format(100 * correct / total))

Accuracy of the model on the test images: 84.7699966430664 %
